In [ ]:
!pip install sentence-transformers PyPDF2  language-tool-python nltk PyMuPDF pinecone langchain openai

In [ ]:
pip install pandas sentence-transformers pinecone-client


In [ ]:
!pip install pinecone

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("pritamdeka/S-PubMedBert-MS-MARCO")

sentences = [
    "bartonellosis",
    "cat scratch disease",
    "cat scratch wound",
    "tick-borne orbivirus fever",
    "cat fur"
]
embeddings = model.encode(sentences)

similarities = model.similarity(embeddings, embeddings)
print(similarities.shape)
# [5, 5]

In [ ]:
generic = "acarbose"

In [ ]:
common = "glucobay"

In [ ]:
embedding_1 = model.encode(generic, show_progress_bar=True)

In [ ]:
embedding_2 = model.encode(common, show_progress_bar=True)

In [ ]:
import numpy as np
from scipy.spatial.distance import cosine, euclidean, cityblock

# Example embeddings (vectors)
#embedding_1 = np.array([0.1, 0.2, 0.3, 0.4, 0.5])
#embedding_2 = np.array([0.5, 0.4, 0.3, 0.2, 0.1])

# Cosine Similarity
def cosine_similarity(embedding_1, embedding_2):
    return 1 - cosine(embedding_1, embedding_2)

# Euclidean Distance (lower distance means higher similarity)
def euclidean_similarity(embedding_1, embedding_2):
    return 1 / (1 + euclidean(embedding_1, embedding_2))  # Scaled to [0, 1]

# Manhattan Distance (also known as L1 distance)
def manhattan_similarity(embedding_1, embedding_2):
    return 1 / (1 + cityblock(embedding_1, embedding_2))  # Scaled to [0, 1]

# Calculate similarities
cos_sim = cosine_similarity(embedding_1, embedding_2)
euclid_sim = euclidean_similarity(embedding_1, embedding_2)
manhattan_sim = manhattan_similarity(embedding_1, embedding_2)

# Display results
print(f"Cosine Similarity: {cos_sim}")
print(f"Euclidean Similarity: {euclid_sim}")
print(f"Manhattan Similarity: {manhattan_sim}")


In [ ]:
import os
from huggingface_hub import InferenceClient

hf_token = os.environ.get("HF_TOKEN")  

if not hf_token:
    raise ValueError("HF_TOKEN environment variable not set!")

client = InferenceClient(api_key=hf_token)

completion = client.chat.completions.create(  
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?"
        }
    ],
)

print(completion.choices[0].message.content)

In [ ]:
import os
api_key = os.environ.get("HF_TOKEN", "")
client = InferenceClient(
    api_key=api_key
)
completion = client.chat.completions.create(
    model="meta-llama/Llama-3.3-70B-Instruct",
     messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": common_name},
    ],
)

content = completion.choices[0].message.content.strip()
content

In [ ]:
import os
from huggingface_hub import InferenceClient
from google.colab import userdata
common_name = "aldara"
system_prompt = f'''You are an medical AI assistant who is an expert in medical drugs data.
 You will be given some common names or brand names of some drugs and you have to give the standardised names of primary drug used in that medicine in lower case. (example1 - given name : Glucobay , answer : acarbose . example 2 - given name : , answer : pregabalin )
 Note : Your output should contain the 2 things. 1) The standardized drug name 2) Justification why you think your answer is correct.
 The format of your response should be exactly like this :
 Drug Name : Standardized drug name
 Justification : Justification for your answer'''
client = InferenceClient(
    api_key=userdata.get('hf_token'),
)
completion = client.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
     messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": common_name},
    ],
)

content = completion.choices[0].message.content.strip()
content

In [ ]:
import os
from google import genai

client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY", ""))
common_name = "sarafem"
system_prompt = f'''You are an medical AI assistant who is an expert in medical drugs data.
 You will be given some common names or brand names of some drugs and you have to give the standardised names of primary drug used in that medicine in lower case. (example1 - given name : Glucobay , answer : acarbose . example 2 - given name : , answer : pregabalin )
 Note : Your output should contain the 2 things. 1) The standardized drug name 2) Justification why you think your answer is correct.
 The format of your response should be exactly like this :
 Drug Name : Standardized drug name
 Justification : Justification for your answer'''
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        {"role": "user", "parts": [{"text": system_prompt + "\n" + common_name}]},
    ],
)

print(response.text)

In [ ]:
import os
pinecone_api = os.environ.get("PINECONE_API_KEY", "")

In [ ]:
import pandas as pd
import os
# Load the CSV file
csv_file = "/content/generic.csv"
df = pd.read_csv(csv_file)

# Specify the column you want to work with
column_name = "genericname"

# Get distinct values, convert to lowercase
distinct_values = df[column_name].dropna().unique()  # Get distinct non-null values

# Convert to lowercase
distinct_values_lower = [value.lower() for value in distinct_values]



In [ ]:
import os
from pinecone import Pinecone, ServerlessSpec


# Step 2: Initialize Pinecone
def initialize_pinecone():
    api_key = pinecone_api
    pc = Pinecone(api_key=api_key, spec=ServerlessSpec(cloud="aws", region="us-east-1"))
    print("DEBUG: Pinecone initialized successfully")  # DEBUG: Added print statement
    return pc

# Example usage
index_name = "dbms"
pc = initialize_pinecone()
index = pc.Index(index_name)

In [ ]:
from sentence_transformers import SentenceTransformer
# Initialize SentenceTransformer model for embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')  # You can use any other model as well



In [ ]:
# Embed the distinct values
embeddings = model.encode(distinct_values_lower)


In [ ]:

# Prepare data for Pinecone (ID, embedding) pairs
pinecone_data = [(str(i), embedding.tolist(), {"word": distinct_values_lower[i]}) for i, embedding in enumerate(embeddings)]


In [ ]:
# Insert the data into Pinecone in batches
batch_size = 200
for i in range(0, len(pinecone_data), batch_size):
    batch = pinecone_data[i : i + batch_size]
    index.upsert(vectors=batch)
    print("uploaded 200 vectors")

print(f"Successfully upserted {len(pinecone_data)} vectors into Pinecone.")

In [ ]:
def get_top_similar_words(query_word, top_n=5):
    # Convert the query word to lowercase and get its embedding
    query_embedding = model.encode([query_word.lower()]).tolist()[0]  # Get the single vector as a list of floats

    # Query Pinecone to get top N most similar vectors (Pinecone returns (id, score) pairs)
    results = index.query(vector=query_embedding, top_k=top_n, include_metadata=True)

    # Extract the top N similar words
    similar_words = []
    for match in results['matches']:
        word = match['metadata'].get('word', 'N/A')  # Extract word from metadata
        score = match['score']
        similar_words.append((word, score))

    return similar_words

# Test the function with a word
query_word = "lisdexamfetamine"  # Replace with the word you want to search for
top_similar_words = get_top_similar_words(query_word)

# Print the results
print(f"Top 5 words similar to '{query_word}':")
for word, score in top_similar_words:
    print(f"Word: {word}, Similarity Score: {score}")

In [ ]:
def get_similar_words_with_threshold(query_word, similarity_threshold=0.5):
    # Convert the query word to lowercase and get its embedding
    query_embedding = model.encode([query_word.lower()]).tolist()[0]  # Get the single vector as a list of floats

    # Query Pinecone to get all similar vectors (without limiting to top_n)
    results = index.query(vector=query_embedding, top_k=1000, include_metadata=True)  # You can set a high `top_k` to get more results

    # Extract the words that have a similarity score greater than the threshold
    similar_words = []
    for match in results['matches']:
        score = match['score']
        if score > similarity_threshold:  # Only keep words with a similarity score > threshold
            word = match['metadata'].get('word', 'N/A')  # Extract word from metadata
            similar_words.append((word, score))

    return similar_words

# Test the function with a word
query_word = "imiquimod"  # Replace with the word you want to search for
threshold = 0.5  # Set your desired threshold for similarity score
similar_words = get_similar_words_with_threshold(query_word, similarity_threshold=threshold)

# Print the results
print(f"Words similar to '{query_word}' with a similarity score greater than {threshold}:")
for word, score in similar_words:
    print(f"Word: {word}, Similarity Score: {score}")


In [ ]:
import os
from huggingface_hub import InferenceClient
from google.colab import userdata
common_name = "lamictal"
choice_prompt = f'''
You are a medical AI assistant and have already identified the standardized drug name for the following common drug name:

Original Drug Name: "aldara"
Standardized Drug Name: "imiquimod"

You are now given a list of drug names in the database based on their similarity to the drug you identified.
Your task is to pick the most accurate one from the list below.

Drug Name Choices:
"imiquimod,imatinib mesylate"

Please choose the most accurate drug name and justify why you picked it.
If There is No correct option , Please return None
Format your response exactly like this:
Chosen Drug: 'chosen_drug_name'
Justification: 'Explanation for why you picked it.'
'''
client = InferenceClient(
    api_key=userdata.get('hf_token'),
)
completion = client.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
     messages=[
        {"role": "system", "content": choice_prompt},
    ],
)

content = completion.choices[0].message.content.strip()
content